In [1]:
#AMBIL DATASET
import cv2
import mediapipe as mp
import os
import csv

mp_hands = mp.solutions.hands # menginisialisasi MediaPipe Hands untuk mendeteksi tangan
hands = mp_hands.Hands(static_image_mode=True)

DATASET_PATH = "dataset_alfabet" 
OUTPUT = "alfabet.csv"

with open(OUTPUT, 'w', newline='') as f: # File CSV dibuka untuk menuliskan hasil ekstraksi
    writer = csv.writer(f)

    for label in os.listdir(DATASET_PATH): # membaca setiap folder dalam dataset
        folder = os.path.join(DATASET_PATH, label)

        if not os.path.isdir(folder):
            continue

        print(f"Processing {label}...")

        for img_name in os.listdir(folder): #Setiap gambar dalam folder dibaca satu per satu untuk diproses.
            path = os.path.join(folder, img_name)

            img = cv2.imread(path)
            if img is None:
                continue

            rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            result = hands.process(rgb)

            if result.multi_hand_landmarks:
                for hand_landmarks in result.multi_hand_landmarks:
                    data = []

                    for lm in hand_landmarks.landmark:
                        data.append(lm.x)
                        data.append(lm.y)

                    data.append(label)
                    writer.writerow(data)

print("✅ alfabet.csv berhasil dibuat!")

Processing A...
Processing B...
Processing C...
Processing D...
Processing del...
Processing E...
Processing F...
Processing G...
Processing H...
Processing I...
Processing J...
Processing K...
Processing L...
Processing M...
Processing N...
Processing nothing...
Processing O...
Processing P...
Processing Q...
Processing R...
Processing S...
Processing space...
Processing T...
Processing U...
Processing V...
Processing W...
Processing X...
Processing Y...
Processing Z...
✅ dataset.csv berhasil dibuat!


In [1]:
#TRAINING MODEL
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib

# Load dataset
df = pd.read_csv('alfabet.csv', header=None)

X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Model
model = RandomForestClassifier()
model.fit(X_train, y_train)

# Evaluasi
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

# Simpan model
joblib.dump(model, 'model_alfabet.pkl')

print(" Model disimpan!")

Accuracy: 0.9646535580524345
 Model disimpan!


In [1]:
#REAL-TIME RECOGNITION
import cv2
import mediapipe as mp
import numpy as np
import joblib

# Load model
model = joblib.load('model.pkl')

# Mediapipe
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

hands = mp_hands.Hands(max_num_hands=1)

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    result = hands.process(rgb)

    if result.multi_hand_landmarks:
        for hand_landmarks in result.multi_hand_landmarks:

            mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

            data = []
            for lm in hand_landmarks.landmark:
                data.append(lm.x)
                data.append(lm.y)

            if len(data) == 42:  # 21 titik x,y
                data = np.array(data).reshape(1, -1)
                pred = model.predict(data)[0]

                cv2.putText(frame, f'Huruf: {pred}', (10, 50),
                            cv2.FONT_HERSHEY_SIMPLEX, 1,
                            (0, 255, 0), 2)

    cv2.imshow("Recognition", frame)

    # tekan Q keluar
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()